# 01 — Data Cleaning

Source: `coffee_orders_dashboard.xlsx` (3 sheets: `orders`, `customers`, `products`), a 2019–2022 transaction
log for a specialty coffee roaster selling Arabica, Excelsa, Liberica, and Robusta beans across the
United States, United Kingdom, and Ireland.

This notebook loads the raw workbook, checks it for the usual data-quality issues (missing values,
duplicates, type mismatches), removes direct personal identifiers, and writes out the clean,
analysis-ready files used by every notebook and SQL script downstream.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

SRC = "../data/raw_data.xlsx"

orders = pd.read_excel(SRC, sheet_name="orders")
customers = pd.read_excel(SRC, sheet_name="customers")
products = pd.read_excel(SRC, sheet_name="products")

print("orders:   ", orders.shape)
print("customers:", customers.shape)
print("products: ", products.shape)


orders:    (1000, 16)
customers: (1000, 9)
products:  (48, 7)


## Schema & missing values

In [2]:
print(orders.dtypes)


Order ID                    str
Order Date       datetime64[us]
Customer ID                 str
Product ID                  str
Quantity                  int64
Customer Name               str
Email                       str
Country                     str
Coffee Type                 str
Roast Type                  str
Size                    float64
Unit Price              float64
Sales                   float64
Coffee Name                 str
Roast Name                  str
Loyalty card                str
dtype: object


In [3]:
missing = orders.isna().sum()
missing = missing[missing > 0]
print("Columns with missing values in `orders`:", len(missing))
print(missing if len(missing) else "None — no missing values in the orders table.")


Columns with missing values in `orders`: 0
None — no missing values in the orders table.


## Duplicate checks

In [4]:
full_dupes = orders.duplicated().sum()
pair_dupes = orders[orders.duplicated(subset=["Order ID", "Product ID"], keep=False)]

print(f"Fully duplicated rows: {full_dupes}")
print(f"Rows sharing the same (Order ID, Product ID): {len(pair_dupes)}")
print(pair_dupes[["Order ID", "Product ID", "Quantity", "Sales"]])


Fully duplicated rows: 0
Rows sharing the same (Order ID, Product ID): 2
          Order ID Product ID  Quantity   Sales
197  NOP-21394-646    L-D-2.5         2  59.570
198  NOP-21394-646    L-D-2.5         3  89.355


The one repeated `(Order ID, Product ID)` pair (`NOP-21394-646` / `L-D-2.5`) has **different
quantities and sales values** on each line (2 units vs. 3 units) — this is two distinct order lines for the
same product within one order, not an exact duplicate, so it is kept as-is rather than dropped.
No exact duplicate rows exist in the dataset.

## Date formatting & referential sanity checks

In [5]:
orders["Order Date"] = pd.to_datetime(orders["Order Date"])
print("Date range:", orders["Order Date"].min().date(), "to", orders["Order Date"].max().date())

recomputed_sales = (orders["Quantity"] * orders["Unit Price"]).round(2)
mismatch = (recomputed_sales - orders["Sales"]).abs() > 0.01
print(f"Rows where Sales != Quantity x Unit Price: {mismatch.sum()} / {len(orders)}")


Date range: 2019-01-02 to 2022-08-19
Rows where Sales != Quantity x Unit Price: 0 / 1000


## Removing personally identifiable information

The raw workbook includes customer name, email, phone number, and postal address — none of which are
needed for sales, retention, or segmentation analysis. Consistent with the data-privacy principle
discussed in the technical report's Responsible AI section, these columns are dropped before the
data is written to any file that ends up in a public repository. Only `Customer ID` (already a
de-identified string key) is retained to join across orders, customers, and products.

In [6]:
orders_clean = orders.drop(columns=["Customer Name", "Email"]).rename(columns={"Loyalty card": "Loyalty Card"})
customers_clean = customers[["Customer ID", "Country", "Loyalty Card"]].drop_duplicates(subset=["Customer ID"])

print("Dropped from orders:    Customer Name, Email")
print("Dropped from customers: Customer Name, Email, Phone Number, Address Line 1, City, Postcode")
print("customers_clean shape:", customers_clean.shape)


Dropped from orders:    Customer Name, Email
Dropped from customers: Customer Name, Email, Phone Number, Address Line 1, City, Postcode
customers_clean shape: (1000, 3)


In [7]:
col_order = [
    "Order ID", "Order Date", "Customer ID", "Product ID", "Quantity",
    "Country", "Coffee Type", "Coffee Name", "Roast Type", "Roast Name",
    "Size", "Unit Price", "Sales", "Loyalty Card",
]
orders_clean = orders_clean[col_order].sort_values(["Order Date", "Order ID"]).reset_index(drop=True)
print(orders_clean.shape)
orders_clean.head(3)


(1000, 14)


## Export cleaned files

In [8]:
orders_clean.to_csv("../data/cleaned_data.csv", index=False)
customers_clean.to_csv("../data/customers_clean.csv", index=False)
products.to_csv("../data/products_clean.csv", index=False)

orders_per_customer = orders_clean.groupby("Customer ID")["Order ID"].nunique()
n_customers = orders_per_customer.shape[0]
n_repeat = (orders_per_customer > 1).sum()

print("Exported cleaned_data.csv, customers_clean.csv, products_clean.csv")
print(f"Unique customers: {n_customers}")
print(f"Customers with more than one order: {n_repeat} ({n_repeat / n_customers:.1%})")


Exported cleaned_data.csv, customers_clean.csv, products_clean.csv
Unique customers: 913
Customers with more than one order: 25 (2.7%)


**Takeaway:** the raw data is unusually clean for a real-world export — no missing values, no exact
duplicate rows, and `Sales` reconciles perfectly with `Quantity x Unit Price` for every line. The main
finding worth carrying into later notebooks is structural, not a data-quality defect: only **25 of 913
customers (2.7%)** ever placed a second order. This has direct consequences for the machine-learning
section (see `03_machine_learning.ipynb`) — a repeat-purchase classifier is fighting severe class
imbalance, not clean signal.